# 06 · External Validation

This notebook applies the four frozen benchmark models, unchanged, to a second public phishing dataset collected by other authors: *Web page phishing detection* (Hannousse and Yahiouche, Mendeley Data, DOI 10.17632/c2gw7fy2j4.2, CC BY 4.0). It follows the protocol frozen in `configs/external-validation.json` before any external result existed. The dataset is downloaded (or an existing snapshot is verified) with the project script, the frozen preparation is applied, every model classifies the primary set, and every published number in `reports/external-validation.json` is recomputed and compared in memory. Nothing is retrained, recalibrated, re-thresholded, or tuned, and no IBM Quantum call is made: the quantum-kernel SVM uses an **exact simulation** of its kernel.

**Inputs**
- `configs/external-validation.json` (the frozen protocol, read only)
- `data/external/hannousse-2020/` (`dataset_B_05_2020.csv`, `acquisition.json`), downloaded by `scripts/download_external.py` if absent
- `data/raw/PhiUSIIL_Phishing_URL_Dataset.csv` (for the overlap step; from notebook 01)
- `configs/benchmark-split.json`, `configs/feature-scaler.json`, `data/processed/benchmark-sample.csv`, `artifacts/benchmark/` (trained models, training angles) from the earlier notebooks
- `reports/external-validation.json`, `reports/benchmark-results.json` (published values to compare against)
- `artifacts/external/predictions.csv` and `external-kernel.npy` (optional, Git-ignored, written by `scripts/evaluate_external.py`; compared when present)

**Outputs**
- `data/external/hannousse-2020/` only if it did not exist yet (the verified download).
- Nothing else on disk. The report is rebuilt in memory; `reports/external-validation.json` is never rewritten.

Run after: `03` (trained models). Notebook `01` provides the PhiUSIIL snapshot.

No host or URL from either dataset is printed: every table below holds counts, ranges, or metrics only.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src").is_dir() and (p / "configs").is_dir()
)
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from scripts import download_external
from scripts import evaluate_external as pipeline
from src.data.benchmark_split import group_hash
from src.evaluation import external
from src.inference import predictor as live

TOL = 1e-9
REPORT_PATH = pipeline.REPORT_PATH  # reports/external-validation.json
ARTIFACT_DIR = pipeline.ARTIFACT_DIR  # artifacts/external (Git-ignored)


def rel(path) -> str:
    # Paths are shown relative to the repository root only.
    return Path(path).resolve().relative_to(ROOT).as_posix()


def read(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


config = read(pipeline.CONFIG_PATH)
dataset = config["dataset"]
published = external.load(REPORT_PATH)  # validated against the report schema
report_bytes = REPORT_PATH.read_bytes()  # checked at the end: this notebook never rewrites it
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", None)
print("Repository root found; working directory set to it.")
print("Protocol:", rel(pipeline.CONFIG_PATH), "|", config["status"], "at", config["frozen_at_utc"])
print("Published report:", rel(REPORT_PATH), "| generated", published["generated_at_utc"])

Repository root found; working directory set to it.
Protocol: configs/external-validation.json | frozen before any external evaluation was run at 2026-10-25T11:10:26+00:00
Published report: reports/external-validation.json | generated 2026-10-25T11:19:54.166113+00:00


## 1. The frozen protocol

The protocol fixes the dataset identity (file, size, SHA-256, row and label counts), the models, the four preparation steps and their order, the metrics, and the bootstrap settings. It was frozen before the evaluation ran, and the published report records the hash of the protocol it followed.

In [2]:
print("Dataset:", dataset["title"], "|", dataset["authors"], "|", dataset["publisher"])
print("DOI", dataset["doi"], "| license", dataset["license"], "| file", dataset["file"])
print(f"Expected: {dataset['expected_rows']} rows, labels {dataset['expected_labels']}")
print("Models evaluated:", ", ".join(config["models"]["evaluated"]))
print("Quantum kernel:", config["models"]["quantum_kernel"])
print("Decision rule:", config["models"]["decision_rule"])
pd.DataFrame(config["preparation"]).set_index("step")

Dataset: Web page phishing detection | Abdelhakim Hannousse and Salima Yahiouche | Mendeley Data
DOI 10.17632/c2gw7fy2j4.2 | license CC BY 4.0 | file dataset_B_05_2020.csv
Expected: 11430 rows, labels {'legitimate': 5715, 'phishing': 5715}
Models evaluated: majority, logistic_regression, rbf_svm, quantum_kernel_svm
Quantum kernel: exact statevector simulation between each external example and the 24 training examples; no QPU
Decision rule: each model's default predict; no threshold, calibration, or retraining


,rule
step,
invalid_host,"derive the host from url with src.inference.predictor.extract_host, which also applies normalize_host (lowercase, drop port and trailing dot, strip one leading www.); exclude rows it rejects"
conflicting_labels,group rows by registrable domain with src.data.benchmark_split.group_key; exclude every group whose rows carry both labels
phiusiil_overlap,"exclude every group that also occurs among the registrable domains of any row of the PhiUSIIL snapshot, so no external domain was available to the benchmark"
duplicate_group,keep the first row of each remaining group in file order; exclude the rest


In [3]:
# Replication check: the report followed this exact protocol, frozen before it was generated.
protocol_sha256 = external.protocol_sha256(pipeline.CONFIG_PATH)
assert published["protocol"] == {
    "path": "configs/external-validation.json",
    "sha256": protocol_sha256,
    "frozen_at_utc": config["frozen_at_utc"],
}, published["protocol"]
assert config["status"] == "frozen before any external evaluation was run"
assert pd.Timestamp(config["frozen_at_utc"]) < pd.Timestamp(published["generated_at_utc"])
assert [step["step"] for step in config["preparation"]] == list(external.STEPS)
assert config["models"]["evaluated"] == list(external.MODELS)
assert config["metrics"]["reported"] == list(external.METRICS)
print("OK: report protocol hash equals configs/external-validation.json:", protocol_sha256[:16])
print("OK: protocol frozen at", config["frozen_at_utc"], "before the report was generated")
print("OK: steps, models and metrics in the code match the protocol, in order")

OK: report protocol hash equals configs/external-validation.json: f1c8250b15e706d6
OK: protocol frozen at 2026-10-25T11:10:26+00:00 before the report was generated
OK: steps, models and metrics in the code match the protocol, in order


## 2. Acquire or verify the dataset

`download_external.acquire()` is the function behind `scripts/download_external.py`. If `data/external/hannousse-2020/` does not exist, it streams the CSV from the documented Mendeley Data URL into a staging directory, checks the size, SHA-256, columns, row and label counts against the protocol, writes `acquisition.json`, and only then moves the snapshot into place. An existing snapshot is reused only if it still passes the same checks; nothing is downloaded again. `pipeline.verified_snapshot` is the offline check that `scripts/evaluate_external.py` runs before evaluating.

In [4]:
manifest, reused = download_external.acquire()
print("Verified existing snapshot; no download." if reused else "Downloaded and verified.")
offline = pipeline.verified_snapshot(dataset)
snapshot = download_external.DEFAULT_DESTINATION
print("Snapshot:", rel(snapshot), "| files:", sorted(p.name for p in snapshot.iterdir()))
pd.Series(
    {
        "bytes": manifest["file"]["bytes"],
        "sha256": manifest["file"]["sha256"],
        "columns": manifest["inspection"]["columns"],
        "rows": manifest["inspection"]["rows"],
        "labels": str(manifest["inspection"]["labels"]),
        "license": manifest["license"]["name"],
    },
    name="acquisition.json",
).to_frame()

Verified existing snapshot; no download.
Snapshot: data/external/hannousse-2020 | files: ['acquisition.json', 'dataset_B_05_2020.csv']


,acquisition.json
bytes,3661166
sha256,21093e2902e5441c86a6daf95e86e7c332046e477fdf109a579d7bd81e586d6c
columns,89
rows,11430
labels,"{'legitimate': 5715, 'phishing': 5715}"
license,CC BY 4.0


In [5]:
# Replication check: the snapshot is the file the protocol names and the report was built from.
assert offline == manifest
assert manifest["file"]["bytes"] == dataset["size_bytes"] == 3661166
assert manifest["file"]["sha256"] == dataset["sha256"] == published["dataset"]["sha256"]
assert manifest["inspection"]["rows"] == dataset["expected_rows"] == published["dataset"]["rows"]
assert manifest["inspection"]["labels"] == dataset["expected_labels"]
assert manifest["doi"] == dataset["doi"] == published["dataset"]["doi"]
assert published["dataset"]["attribution"] == manifest["attribution"]
assert published["dataset"]["license"] == dataset["license"] == "CC BY 4.0"
print(f"OK: {dataset['file']} has {dataset['size_bytes']} bytes and the protocol's SHA-256")
print("OK: 11,430 rows, 5,715 legitimate and 5,715 phishing, as the protocol states")
print("OK: DOI, SHA-256, row count, license and attribution equal the published report")

OK: dataset_B_05_2020.csv has 3661166 bytes and the protocol's SHA-256
OK: 11,430 rows, 5,715 legitimate and 5,715 phishing, as the protocol states
OK: DOI, SHA-256, row count, license and attribution equal the published report


## 3. Preparation funnel

`pipeline.load_inputs` loads the benchmark inputs (it checks the split manifest and scaler hashes the models were trained on), reads only the `url` and `status` columns, collects the registrable-domain groups of every PhiUSIIL row (after checking the PhiUSIIL checksum), and applies the four frozen steps in order with `external.prepare`. Each step removes rows from what the previous step left.

In [6]:
data, prepared, scaler = pipeline.load_inputs(dataset)
remaining = prepared.input_rows
funnel = []
for entry in prepared.exclusions:
    remaining -= entry["rows"]
    funnel.append(
        {
            "step": entry["step"],
            "rows excluded": entry["rows"],
            "groups excluded": entry.get("groups", ""),
            "rows remaining": remaining,
        }
    )
counts = prepared.class_counts
share = counts["phishing"] / len(prepared.rows)
print(f"Input rows: {prepared.input_rows}")
print(f"Primary set: {len(prepared.rows)} rows, {counts} ({100 * share:.1f}% phishing)")
pd.DataFrame(funnel).set_index("step")

Input rows: 11430
Primary set: 6441 rows, {'legitimate': 3470, 'phishing': 2971} (46.1% phishing)


,rows excluded,groups excluded,rows remaining
step,,,
invalid_host,0,,11430
conflicting_labels,397,17,11033
phiusiil_overlap,1805,802,9228
duplicate_group,2787,,6441


In [7]:
# Replication check: the funnel equals the published one, and the primary set obeys every rule.
assert prepared.exclusions == published["preparation"]["exclusions"], prepared.exclusions
assert prepared.input_rows == published["preparation"]["input_rows"] == 11430
assert len(prepared.rows) == published["preparation"]["primary_rows"] == remaining == 6441
assert counts == published["preparation"]["class_counts"] == {"legitimate": 3470, "phishing": 2971}
print("OK: rows and groups excluded by each step equal the published report (exact)")
print("OK: primary set 6,441 rows (3,470 legitimate, 2,971 phishing), as published")

# Independent checks on the primary set itself.
groups = prepared.rows["group"]
assert prepared.rows["host"].notna().all() and groups.notna().all()
assert groups.is_unique, "a registrable domain appears twice"
assert not set(groups) & pipeline.load_phiusiil_groups(), "a PhiUSIIL domain survived"
assert prepared.rows["row_index"].is_monotonic_increasing  # first row of each group, file order
assert published["benchmark"]["input_hashes"] == data.input_hashes
assert published["benchmark"]["model_version"] == live.Predictor.from_artifacts().model_version
assert published["benchmark"]["training_examples"] == len(data.train_labels) == 24
assert published["benchmark"]["benchmark_test_examples"] == len(data.test_labels) == 12
print("OK: one row per registrable domain, none of them present anywhere in PhiUSIIL")
print("OK: benchmark input hashes and model version equal the published report (frozen models)")

OK: rows and groups excluded by each step equal the published report (exact)
OK: primary set 6,441 rows (3,470 legitimate, 2,971 phishing), as published


OK: one row per registrable domain, none of them present anywhere in PhiUSIIL
OK: benchmark input hashes and model version equal the published report (frozen models)


## 4. Features and clipping

The four host features are computed with the benchmark's own code and scaled with the unchanged training scaler. The scaler was fitted on 24 PhiUSIIL hosts, so a value outside their range is clipped to the nearest bound (angle 0 or pi/2) and counted. The table shows the training range next to aggregate statistics of the external values; no individual host is shown.

In [8]:
raw, angles, clipped = external.featurize(prepared.rows["host"], scaler)
low, high = np.array(scaler.minima), np.array(scaler.maxima)
below, above = (raw < low).sum(axis=0), (raw > high).sum(axis=0)
features = pd.DataFrame(
    {
        "training min": low,
        "training max": high,
        "external min": raw.min(axis=0),
        "external median": np.median(raw, axis=0),
        "external max": raw.max(axis=0),
        "below range": below,
        "above range": above,
        "clipped": [clipped["per_feature"][name] for name in scaler.features],
    },
    index=list(scaler.features),
)
any_clipped = clipped["rows_with_any_clipped"]
print(
    f"Rows with at least one clipped feature: {any_clipped} of {len(raw)} "
    f"({100 * any_clipped / len(raw):.1f}%)"
)
features.round(4)

Rows with at least one clipped feature: 600 of 6441 (9.3%)


,training min,training max,external min,external median,external max,below range,above range,clipped
host_entropy,2.3219,4.5739,1.7925,3.4183,5.0553,47,12,59
host_label_count,2.0000,3.0000,2.0000,2.0000,13.0000,0,396,396
host_digit_ratio,0.0000,0.1558,0.0000,0.0000,0.8000,0,317,317
host_hyphen_count,0.0000,6.0000,0.0000,0.0000,19.0000,0,3,3


In [9]:
# Replication check: clipping counts equal the published ones; angles stay in [0, pi/2].
assert clipped == published["clipped_values"], clipped
assert not any(scaler.constant), "the frozen scaler has no constant feature"
assert (below + above).tolist() == [clipped["per_feature"][n] for n in scaler.features]
assert int(((raw < low) | (raw > high)).any(axis=1).sum()) == any_clipped == 600
assert angles.shape == (6441, 4) and angles.min() >= 0 and angles.max() <= np.pi / 2
print("OK: clipped values per feature", clipped["per_feature"], "equal the published report")
print("OK: an independent count of values outside the training range gives the same numbers")
print("OK: 600 rows have at least one clipped feature; all angles lie in [0, pi/2]")

OK: clipped values per feature {'host_entropy': 59, 'host_label_count': 396, 'host_digit_ratio': 317, 'host_hyphen_count': 3} equal the published report
OK: an independent count of values outside the training range gives the same numbers
OK: 600 rows have at least one clipped feature; all angles lie in [0, pi/2]


## 5. Classification and bootstrap intervals

`pipeline.classify` simulates the external kernel exactly (each of the 6,441 examples against the 24 training examples; statevector, no QPU) and classifies the primary set with each model's default `predict`: the quantum-kernel SVM receives the 6,441 x 24 kernel, the classical models the scaled angles. `pipeline.with_intervals` then draws 2,000 bootstrap resamples of rows with seed 42, the same rows for every model, and takes the 2.5th and 97.5th percentiles of each metric.

In [10]:
labels = prepared.rows["is_phishing"].to_numpy()
evaluations, kernel, kernel_seconds = pipeline.classify(data, angles, labels)
print(f"External kernel {kernel.shape}: exact simulation in {kernel_seconds:.1f} s (no QPU).")
settings = config["metrics"]["confidence_intervals"]
rows, used = pipeline.with_intervals(evaluations, labels, settings)
print(f"Bootstrap: {settings['resamples']} resamples, seed {settings['seed']}, {used} used.")


def span(interval):
    return "n/a" if interval is None else f"{interval[0]:.3f}-{interval[1]:.3f}"


table = {}
for row in rows:
    entry = {}
    for name in external.METRICS:
        value = row["metrics"][name]
        point = "n/a" if value is None else f"{value:.3f}"
        entry[name] = f"{point} ({span(row['confidence_intervals'][name])})"
    entry["TN / FP / FN / TP"] = " / ".join(str(v) for r in row["confusion_matrix"] for v in r)
    table[f"{row['name']} ({row['kernel_source']})"] = entry
pd.DataFrame(table).T

External kernel (6441, 24): exact simulation in 2.1 s (no QPU).


Bootstrap: 2000 resamples, seed 42, 2000 used.


,accuracy,balanced_accuracy,precision,recall,f1,average_precision,TN / FP / FN / TP
Majority baseline (classical),0.539 (0.526-0.551),0.500 (0.500-0.500),0.000 (0.000-0.000),0.000 (0.000-0.000),0.000 (0.000-0.000),n/a (n/a),3470 / 0 / 2971 / 0
Logistic regression (classical),0.663 (0.651-0.674),0.653 (0.642-0.665),0.669 (0.651-0.689),0.530 (0.513-0.548),0.592 (0.577-0.607),0.670 (0.653-0.687),2692 / 778 / 1395 / 1576
RBF SVM (classical),0.644 (0.632-0.656),0.637 (0.626-0.649),0.629 (0.611-0.647),0.555 (0.538-0.572),0.590 (0.575-0.604),0.647 (0.629-0.666),2497 / 973 / 1322 / 1649
Quantum-kernel SVM (simulation),0.522 (0.510-0.535),0.528 (0.516-0.541),0.486 (0.469-0.502),0.611 (0.594-0.629),0.541 (0.527-0.556),0.461 (0.445-0.479),1546 / 1924 / 1155 / 1816


In [11]:
# Replication check: every model equals the published report (metrics 1e-9, CMs and CIs exact).
assert [row["model"] for row in published["models"]] == list(external.MODELS)
assert used == published["bootstrap"]["resamples_used"] == 2000
for key in ("level", "resamples", "seed"):
    assert published["bootstrap"][key] == settings[key], key
assert published["bootstrap"]["method"] == external.BOOTSTRAP_METHOD
for mine, theirs in zip(rows, published["models"], strict=True):
    where = mine["model"]
    for key in ("name", "model", "kernel_source", "confusion_matrix", "confidence_intervals"):
        assert mine[key] == theirs[key], (where, key)
    for name in external.METRICS:
        a, b = mine["metrics"][name], theirs["metrics"][name]
        assert (a is None) == (b is None) and (a is None or abs(a - b) <= TOL), (where, name)
    print(
        f"OK: {theirs['name']:<20} ({theirs['kernel_source']}) metrics (1e-9), CM "
        f"{mine['confusion_matrix']} and intervals (exact) equal the report"
    )
assert published["models"][-1]["kernel_source"] == "simulation"
print("OK: the quantum-kernel SVM row is labeled simulation; no hardware row exists")

OK: Majority baseline    (classical) metrics (1e-9), CM [[3470, 0], [2971, 0]] and intervals (exact) equal the report
OK: Logistic regression  (classical) metrics (1e-9), CM [[2692, 778], [1395, 1576]] and intervals (exact) equal the report
OK: RBF SVM              (classical) metrics (1e-9), CM [[2497, 973], [1322, 1649]] and intervals (exact) equal the report
OK: Quantum-kernel SVM   (simulation) metrics (1e-9), CM [[1546, 1924], [1155, 1816]] and intervals (exact) equal the report
OK: the quantum-kernel SVM row is labeled simulation; no hardware row exists


In [12]:
# Replication check: metrics recomputed directly with scikit-learn, outside evaluate();
# the bootstrap's own metric function gives the same point values.
for row in rows:
    evaluation = evaluations[row["model"]]
    predicted = np.array(evaluation["predictions"])
    scores = evaluation["decision_scores"]
    direct = {
        "accuracy": accuracy_score(labels, predicted),
        "balanced_accuracy": balanced_accuracy_score(labels, predicted),
        "precision": precision_score(labels, predicted, zero_division=0),
        "recall": recall_score(labels, predicted, zero_division=0),
        "f1": f1_score(labels, predicted, zero_division=0),
        "average_precision": None if scores is None else average_precision_score(labels, scores),
    }
    own = external.point_metrics(labels, predicted, None if scores is None else np.array(scores))
    for name, value in direct.items():
        for other in (row["metrics"][name], own[name]):
            assert (value is None and other is None) or abs(value - other) <= TOL, name
    assert confusion_matrix(labels, predicted, labels=[0, 1]).tolist() == row["confusion_matrix"]
print("OK: scikit-learn metrics and confusion matrices equal the reported ones (1e-9)")
print("OK: the bootstrap metric function reproduces every point estimate (1e-9)")
assert kernel.shape == (6441, 24) and kernel.min() >= -TOL and kernel.max() <= 1 + TOL
print("OK: the simulated kernel is 6,441 x 24 with values in [0, 1]")

OK: scikit-learn metrics and confusion matrices equal the reported ones (1e-9)
OK: the bootstrap metric function reproduces every point estimate (1e-9)
OK: the simulated kernel is 6,441 x 24 with values in [0, 1]


In [13]:
# Optional check: the per-row artifacts written by scripts/evaluate_external.py, if present.
saved_csv, saved_kernel = ARTIFACT_DIR / "predictions.csv", ARTIFACT_DIR / "external-kernel.npy"
if saved_csv.is_file() and saved_kernel.is_file():
    saved = pd.read_csv(saved_csv)
    assert saved["row_index"].tolist() == prepared.rows["row_index"].tolist()
    assert saved["group_hash"].tolist() == prepared.rows["group"].map(group_hash).tolist()
    assert saved["is_phishing"].tolist() == labels.tolist()
    for name, evaluation in evaluations.items():
        assert saved[f"{name}_prediction"].tolist() == evaluation["predictions"], name
        if evaluation["decision_scores"] is not None:
            assert np.allclose(saved[f"{name}_score"], evaluation["decision_scores"], atol=TOL)
    assert np.allclose(np.load(saved_kernel), kernel, rtol=0, atol=TOL)
    print(f"OK: {rel(saved_csv)} has the same rows, labels, predictions and scores (1e-9)")
    print(f"OK: {rel(saved_kernel)} equals the kernel simulated here (1e-9)")
else:
    print("Per-row artifacts not found; run scripts/evaluate_external.py to compare them.")

OK: artifacts/external/predictions.csv has the same rows, labels, predictions and scores (1e-9)
OK: artifacts/external/external-kernel.npy equals the kernel simulated here (1e-9)


## 6. Comparison with the 12-example benchmark

The benchmark test set has 12 examples, so each one moves accuracy by about 8.3 percentage points and no interval was published for it. The external primary set has 6,441 rows; its 95% accuracy intervals are 2.3 to 2.5 points wide. For the quantum-kernel SVM both columns use the exact simulated kernel (on the benchmark, the IBM hardware kernels gave the same 12 predictions).

In [14]:
benchmark = read("reports/benchmark-results.json")
bench_rows = {(m["name"], m["kernel_source"]): m for m in benchmark["models"]}
comparison = {}
for row in rows:
    bench = bench_rows[(row["name"], row["kernel_source"])]["metrics"]
    ci = row["confidence_intervals"]
    comparison[f"{row['name']} ({row['kernel_source']})"] = {
        "benchmark accuracy (12)": f"{bench['accuracy']:.3f}",
        "external accuracy (6,441)": f"{row['metrics']['accuracy']:.3f} ({span(ci['accuracy'])})",
        "benchmark balanced acc. (12)": f"{bench['balanced_accuracy']:.3f}",
        "external balanced acc. (6,441)": (
            f"{row['metrics']['balanced_accuracy']:.3f} ({span(ci['balanced_accuracy'])})"
        ),
    }
print(f"Phishing share of the primary set: {share:.3f} (average precision of random ranking)")
pd.DataFrame(comparison).T

Phishing share of the primary set: 0.461 (average precision of random ranking)


,benchmark accuracy (12),"external accuracy (6,441)",benchmark balanced acc. (12),"external balanced acc. (6,441)"
Majority baseline (classical),0.500,0.539 (0.526-0.551),0.500,0.500 (0.500-0.500)
Logistic regression (classical),0.833,0.663 (0.651-0.674),0.833,0.653 (0.642-0.665)
RBF SVM (classical),0.917,0.644 (0.632-0.656),0.917,0.637 (0.626-0.649)
Quantum-kernel SVM (simulation),0.500,0.522 (0.510-0.535),0.500,0.528 (0.516-0.541)


In [15]:
# Replication check: the values quoted in the Interpretation below.
by_model = {row["model"]: row for row in rows}
quoted = {
    ("logistic_regression", "balanced_accuracy"): "0.653",
    ("rbf_svm", "balanced_accuracy"): "0.637",
    ("logistic_regression", "accuracy"): "0.663",
    ("rbf_svm", "accuracy"): "0.644",
    ("quantum_kernel_svm", "accuracy"): "0.522",
    ("quantum_kernel_svm", "balanced_accuracy"): "0.528",
    ("quantum_kernel_svm", "average_precision"): "0.461",
    ("majority", "accuracy"): "0.539",
}
for (model, metric), text in quoted.items():
    assert f"{by_model[model]['metrics'][metric]:.3f}" == text, (model, metric)
assert f"{share:.3f}" == "0.461" and f"{100 * any_clipped / len(raw):.1f}" == "9.3"
for model in ("logistic_regression", "rbf_svm"):
    assert by_model[model]["confidence_intervals"]["balanced_accuracy"][0] > 0.5, model
widths = [
    by_model[m]["confidence_intervals"]["accuracy"][1]
    - by_model[m]["confidence_intervals"]["accuracy"][0]
    for m in external.MODELS
]
assert (f"{100 * min(widths):.1f}", f"{100 * max(widths):.1f}") == ("2.3", "2.5"), widths
lr_ci = by_model["logistic_regression"]["confidence_intervals"]["accuracy"]
rbf_ci = by_model["rbf_svm"]["confidence_intervals"]["accuracy"]
assert lr_ci[0] <= rbf_ci[1] and rbf_ci[0] <= lr_ci[1], "accuracy intervals no longer overlap"
q_ci = by_model["quantum_kernel_svm"]["confidence_intervals"]["accuracy"]
m_ci = by_model["majority"]["confidence_intervals"]["accuracy"]
assert q_ci[1] >= m_ci[0] and m_ci[1] >= q_ci[0], "quantum and majority intervals separate"
for name, source, accuracy in (
    ("Logistic regression", "classical", "0.833"),
    ("RBF SVM", "classical", "0.917"),
    ("Quantum-kernel SVM", "simulation", "0.500"),
):
    assert f"{bench_rows[(name, source)]['metrics']['accuracy']:.3f}" == accuracy, name
print("OK: classical balanced accuracies 0.653 and 0.637, both intervals above 0.5")
print("OK: logistic regression and RBF SVM accuracy intervals overlap")
print("OK: quantum-kernel SVM accuracy 0.522, balanced 0.528, AP 0.461 = phishing share 0.461")
print("OK: quantum-kernel SVM and majority (0.539) accuracy intervals overlap")
print("OK: external accuracy intervals are 2.3 to 2.5 points wide; 9.3% of rows clipped")
print("OK: benchmark accuracies 0.833, 0.917 and 0.500 (simulation)")

assert REPORT_PATH.read_bytes() == report_bytes
print("OK: reports/external-validation.json was not modified by this notebook")

OK: classical balanced accuracies 0.653 and 0.637, both intervals above 0.5
OK: logistic regression and RBF SVM accuracy intervals overlap
OK: quantum-kernel SVM accuracy 0.522, balanced 0.528, AP 0.461 = phishing share 0.461
OK: quantum-kernel SVM and majority (0.539) accuracy intervals overlap
OK: external accuracy intervals are 2.3 to 2.5 points wide; 9.3% of rows clipped
OK: benchmark accuracies 0.833, 0.917 and 0.500 (simulation)
OK: reports/external-validation.json was not modified by this notebook


## 7. Interpretation

- **The classical models keep a real but much smaller signal outside PhiUSIIL.** Logistic regression and the RBF SVM reach balanced accuracies of 0.653 and 0.637 on 6,441 external rows, and both intervals lie above the 0.5 of chance. That is far below their 0.833 and 0.917 on the 12 benchmark test examples, where accuracy and balanced accuracy coincide because the test set is balanced. The benchmark figure came from 12 rows, where one example is worth 8.3 points; each external accuracy interval is 2.3 to 2.5 points wide. The gap has several possible causes that this validation does not separate: the tiny benchmark test set, the distribution shift between the datasets (9.3% of external rows fall outside the training range on at least one feature), and collection artifacts in PhiUSIIL that may have made the benchmark easier.
- **The external data do not rank the two classical models.** Their accuracies (0.663 and 0.644) have overlapping intervals, so the order seen on the benchmark is not confirmed.
- **The quantum-kernel SVM is near chance, as on the benchmark.** Its balanced accuracy is 0.528, and its average precision (0.461) equals the phishing share of the primary set, so its scores do not rank phishing hosts above legitimate ones. Its accuracy (0.522) is not above the majority baseline's (0.539, which only reflects the legitimate share), and the two intervals overlap. This agrees with its 6/12 on the benchmark test set, in simulation and on hardware.
- **Scope.** These are the frozen models, trained on 24 examples with four host-only features; the result says nothing about the method with more data or richer features. The quantum kernel was simulated exactly, so hardware noise is not included. **No claim of quantum advantage follows.**